
# Numpy Backend Example: Matching Image Keypoints by QAP Solvers

This example shows how to match image keypoints by graph matching solvers provided by ``pygmtools``.
These solvers follow the Quadratic Assignment Problem formulation and can generally work out-of-box.
The matched images can be further processed for other downstream tasks.


In [ ]:
# Author: Runzhong Wang <runzhong.wang@sjtu.edu.cn>
#         Wenzheng Pan <pwz1121@sjtu.edu.cn>
#
# License: Mulan PSL v2 License

<div class="alert alert-info"><h4>Note</h4><p>The following solvers support QAP formulation, and are included in this example:

    * :func:`~pygmtools.classic_solvers.rrwm` (classic solver)

    * :func:`~pygmtools.classic_solvers.ipfp` (classic solver)

    * :func:`~pygmtools.classic_solvers.sm` (classic solver)

    * :func:`~pygmtools.neural_solvers.ngm` (neural network solver)</p></div>




In [ ]:
import numpy as np # numpy backend
import cv2 as cv
import pygmtools as pygm
import matplotlib.pyplot as plt # for plotting
from matplotlib.patches import ConnectionPatch # for plotting matching result
import scipy.io as sio # for loading .mat file
import scipy.spatial as spa # for Delaunay triangulation
from sklearn.decomposition import PCA as PCAdimReduc
import itertools
from PIL import Image
pygm.set_backend('numpy') # set numpy as backend for pygmtools

## Load the images
Images are from the Willow Object Class dataset (this dataset also available with the Benchmark of ``pygmtools``,
see :class:`~pygmtools.dataset.WillowObject`).

The images are resized to 256x256.




In [ ]:
# import pygmtools as pygm
# # from pygm.benchmark import Benchmark

# # Define Benchmark on PascalVOC.
# bm = pygm.benchmark.Benchmark(name='PascalVOC', sets='test',
#                obj_resize=(256, 256), problem='2GM',
#                filter='intersection')

# # Random fetch data and ground truth.
# data_list, gt_dict, _ = bm.get_data(ids=["2008_002773_8_aeroplane", "2011_002930_1_aeroplane"])
# img1 = data_list[0]['img']
# img2 = data_list[1]['img']
# kpts1_x, kpts1_y = [], []
# kpts2_x, kpts2_y = [], []
# kpts1_x = [kpt['x'] for kpt in data_list[0]['kpts']]
# kpts1_y = [kpt['y'] for kpt in data_list[0]['kpts']]
# kpts2_x = [kpt['x'] for kpt in data_list[1]['kpts']]
# kpts2_y = [kpt['y'] for kpt in data_list[1]['kpts']]
# kpts1 = np.array([kpts1_x, kpts1_y])
# kpts2 = np.array([kpts2_x, kpts2_y])


In [ ]:
obj_resize = (256, 256)
# img1 = Image.open('../data/willow_duck_0001.png')
# img2 = Image.open('../data/willow_duck_0002.png')
# kpts1 = np.array(sio.loadmat('../data/willow_duck_0001.mat')['pts_coord'])
# kpts2 = np.array(sio.loadmat('../data/willow_duck_0002.mat')['pts_coord'])
img1 = Image.open('/home/xjx/A-xjx/QAPs/baselines/data/WILLOW-ObjectClass_dataset/WILLOW-ObjectClass/Car/Cars_000a.png')
img2 = Image.open('/home/xjx/A-xjx/QAPs/baselines/data/WILLOW-ObjectClass_dataset/WILLOW-ObjectClass/Car/Cars_006b.png')
kpts1 = np.array(sio.loadmat('/home/xjx/A-xjx/QAPs/baselines/data/WILLOW-ObjectClass_dataset/WILLOW-ObjectClass/Car/Cars_000a.mat')['pts_coord'])
kpts2 = np.array(sio.loadmat('/home/xjx/A-xjx/QAPs/baselines/data/WILLOW-ObjectClass_dataset/WILLOW-ObjectClass/Car/Cars_006b.mat')['pts_coord'])
kpts1[0] = kpts1[0] * obj_resize[0] / img1.size[0]
kpts1[1] = kpts1[1] * obj_resize[1] / img1.size[1]
kpts2[0] = kpts2[0] * obj_resize[0] / img2.size[0]
kpts2[1] = kpts2[1] * obj_resize[1] / img2.size[1]
img1 = img1.resize(obj_resize, resample=Image.BILINEAR)
img2 = img2.resize(obj_resize, resample=Image.BILINEAR)

Visualize the images and keypoints




In [ ]:
def plot_image_with_graph(img, kpt, A=None):
    plt.imshow(img)
    plt.scatter(kpt[0], kpt[1], c='w', edgecolors='k')
    if A is not None:
        for x, y in zip(np.nonzero(A)[0], np.nonzero(A)[1]):
            plt.plot((kpt[0, x], kpt[0, y]), (kpt[1, x], kpt[1, y]), 'k-')

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.title('Image 1')
plot_image_with_graph(img1, kpts1)
plt.subplot(1, 2, 2)
plt.title('Image 2')
plot_image_with_graph(img2, kpts2)

## Build the graphs
Graph structures are built based on the geometric structure of the keypoint set. In this example,
we refer to [Delaunay triangulation](https://en.wikipedia.org/wiki/Delaunay_triangulation).




In [ ]:
kpts1

In [ ]:
def delaunay_triangulation(kpt):
    d = spa.Delaunay(kpt.T)
    A = np.zeros((len(kpt[0]), len(kpt[0])))
    for simplex in d.simplices:
        for pair in itertools.permutations(simplex, 2):
            A[pair] = 1
    return A

A1 = delaunay_triangulation(kpts1)
A2 = delaunay_triangulation(kpts2)

In [ ]:
A2

We encode the length of edges as edge features




In [ ]:
A1 = ((np.expand_dims(kpts1, 1) - np.expand_dims(kpts1, 2)) ** 2).sum(axis=0) * A1
A1 = (A1 / A1.max()).astype(np.float32)
A2 = ((np.expand_dims(kpts2, 1) - np.expand_dims(kpts2, 2)) ** 2).sum(axis=0) * A2
A2 = (A2 / A2.max()).astype(np.float32)

In [ ]:
A1, A2

Visualize the graphs




In [ ]:
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.title('Image 1 with Graphs')
plot_image_with_graph(img1, kpts1, A1)
plt.subplot(1, 2, 2)
plt.title('Image 2 with Graphs')
plot_image_with_graph(img2, kpts2, A2)

## Extract node features
Let's adopt the SIFT method to extract node features.




In [ ]:
np_img1 = np.array(img1, dtype=np.float32)
np_img2 = np.array(img2, dtype=np.float32)

def detect_sift(img):
    sift = cv.SIFT_create() 
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    img8bit = cv.normalize(gray, None, 0, 255, cv.NORM_MINMAX).astype('uint8')
    kpt = sift.detect(img8bit, None) 
    kpt, feat = sift.compute(img8bit, kpt) 
    return kpt, feat

sift_kpts1, feat1 = detect_sift(np_img1)
sift_kpts2, feat2 = detect_sift(np_img2)
sift_kpts1 = np.round(cv.KeyPoint_convert(sift_kpts1).T).astype(int)
sift_kpts2 = np.round(cv.KeyPoint_convert(sift_kpts2).T).astype(int)

Normalize the features




In [ ]:
num_features = feat1.shape[1]
feat1 = feat1 / np.expand_dims(np.linalg.norm(feat1, axis=1), 1).repeat(128, axis=1)
feat2 = feat2 / np.expand_dims(np.linalg.norm(feat2, axis=1), 1).repeat(128, axis=1)

Extract node features by nearest interpolation




In [ ]:
rounded_kpts1 = np.round(kpts1).astype(int)
rounded_kpts2 = np.round(kpts2).astype(int)

idx_1, idx_2 = [], []
for i in range(rounded_kpts1.shape[1]):
    y1 = np.where(sift_kpts1[1] == sift_kpts1[1][np.abs(sift_kpts1[1] - rounded_kpts1[1][i]).argmin()])
    y2 = np.where(sift_kpts2[1] == sift_kpts2[1][np.abs(sift_kpts2[1] - rounded_kpts2[1][i]).argmin()])
    t1 = sift_kpts1[0][y1]
    t2 = sift_kpts2[0][y2]
    x1 = np.where(sift_kpts1[0] == t1[np.abs(t1 - rounded_kpts1[0][i]).argmin()])
    x2 = np.where(sift_kpts2[0] == t2[np.abs(t2 - rounded_kpts2[0][i]).argmin()])
    idx_1.append(np.intersect1d(x1, y1)[0])
    idx_2.append(np.intersect1d(x2, y2)[0])

node1 = feat1[idx_1, :] # shape: NxC
node2 = feat2[idx_2, :] # shape: NxC

## Build affinity matrix
We follow the formulation of Quadratic Assignment Problem (QAP):

\begin{align}&\max_{\mathbf{X}} \ \texttt{vec}(\mathbf{X})^\top \mathbf{K} \texttt{vec}(\mathbf{X})\\
    s.t. \quad &\mathbf{X} \in \{0, 1\}^{n_1\times n_2}, \ \mathbf{X}\mathbf{1} = \mathbf{1}, \ \mathbf{X}^\top\mathbf{1} \leq \mathbf{1}\end{align}

where the first step is to build the affinity matrix ($\mathbf{K}$)




In [ ]:
conn1, edge1 = pygm.utils.dense_to_sparse(A1)
conn2, edge2 = pygm.utils.dense_to_sparse(A2)
import functools
gaussian_aff = functools.partial(pygm.utils.gaussian_aff_fn, sigma=1) # set affinity function
K = pygm.utils.build_aff_mat(node1, edge1, conn1, node2, edge2, conn2, edge_aff_fn=gaussian_aff)

In [ ]:
print(K.shape)

Visualization of the affinity matrix. For graph matching problem with $N$ nodes, the affinity matrix
has $N^2\times N^2$ elements because there are $N^2$ edges in each graph.

<div class="alert alert-info"><h4>Note</h4><p>The diagonal elements are node affinities, the off-diagonal elements are edge features.</p></div>




In [ ]:
plt.figure(figsize=(4, 4))
plt.title(f'Affinity Matrix (size: {K.shape[0]}$\\times${K.shape[1]})')
plt.imshow(K, cmap='Blues')

## Solve graph matching problem by RRWM solver
See :func:`~pygmtools.classic_solvers.rrwm` for the API reference.




In [ ]:
X = pygm.rrwm(K, kpts1.shape[1], kpts2.shape[1])

The output of RRWM is a soft matching matrix. Hungarian algorithm is then adopted to reach a discrete matching matrix.




In [ ]:
X = pygm.hungarian(X)

## Plot the matching
The correct matchings are marked by green, and wrong matchings are marked by red. In this example, the nodes are
ordered by their ground truth classes (i.e. the ground truth matching matrix is a diagonal matrix).




In [ ]:
plt.figure(figsize=(8, 4))
plt.suptitle('Image Matching Result by RRWM')
ax1 = plt.subplot(1, 2, 1)
plot_image_with_graph(img1, kpts1, A1)
ax2 = plt.subplot(1, 2, 2)
plot_image_with_graph(img2, kpts2, A2)
for i in range(X.shape[0]):
    j = np.argmax(X[i]).item()
    con = ConnectionPatch(xyA=kpts1[:, i], xyB=kpts2[:, j], coordsA="data", coordsB="data",
                          axesA=ax1, axesB=ax2, color="red" if i != j else "green")
    plt.gca().add_artist(con)

## Solve by other solvers
We could also do a quick benchmarking of other solvers on this specific problem.

### IPFP solver
See :func:`~pygmtools.classic_solvers.ipfp` for the API reference.




In [ ]:
X = pygm.ipfp(K, kpts1.shape[1], kpts2.shape[1])

plt.figure(figsize=(8, 4))
plt.suptitle('Image Matching Result by IPFP')
ax1 = plt.subplot(1, 2, 1)
plot_image_with_graph(img1, kpts1, A1)
ax2 = plt.subplot(1, 2, 2)
plot_image_with_graph(img2, kpts2, A2)
for i in range(X.shape[0]):
    j = np.argmax(X[i]).item()
    con = ConnectionPatch(xyA=kpts1[:, i], xyB=kpts2[:, j], coordsA="data", coordsB="data",
                          axesA=ax1, axesB=ax2, color="red" if i != j else "green")
    plt.gca().add_artist(con)

### SM solver
See :func:`~pygmtools.classic_solvers.sm` for the API reference.




In [ ]:
X = pygm.sm(K, kpts1.shape[1], kpts2.shape[1])
X = pygm.hungarian(X)

plt.figure(figsize=(8, 4))
plt.suptitle('Image Matching Result by SM')
ax1 = plt.subplot(1, 2, 1)
plot_image_with_graph(img1, kpts1, A1)
ax2 = plt.subplot(1, 2, 2)
plot_image_with_graph(img2, kpts2, A2)
for i in range(X.shape[0]):
    j = np.argmax(X[i]).item()
    con = ConnectionPatch(xyA=kpts1[:, i], xyB=kpts2[:, j], coordsA="data", coordsB="data",
                          axesA=ax1, axesB=ax2, color="red" if i != j else "green")
    plt.gca().add_artist(con)

### NGM solver
See :func:`~pygmtools.neural_solvers.ngm` for the API reference.

<div class="alert alert-info"><h4>Note</h4><p>The NGM solvers are pretrained on a different problem setting, so their performance may seem inferior.
    To improve their performance, you may change the way of building affinity matrices, or try finetuning
    NGM on the new problem.</p></div>

The NGM solver pretrained on Willow dataset:




In [ ]:
X = pygm.ngm(K, kpts1.shape[1], kpts2.shape[1], pretrain='willow')
X = pygm.hungarian(X)

plt.figure(figsize=(8, 4))
plt.suptitle('Image Matching Result by NGM (willow pretrain)')
ax1 = plt.subplot(1, 2, 1)
plot_image_with_graph(img1, kpts1, A1)
ax2 = plt.subplot(1, 2, 2)
plot_image_with_graph(img2, kpts2, A2)
for i in range(X.shape[0]):
    j = np.argmax(X[i]).item()
    con = ConnectionPatch(xyA=kpts1[:, i], xyB=kpts2[:, j], coordsA="data", coordsB="data",
                          axesA=ax1, axesB=ax2, color="red" if i != j else "green")
    plt.gca().add_artist(con)

The NGM solver pretrained on VOC dataset:




In [ ]:
X = pygm.ngm(K, kpts1.shape[1], kpts2.shape[1], pretrain='voc')
X = pygm.hungarian(X)

plt.figure(figsize=(8, 4))
plt.suptitle('Image Matching Result by NGM (voc pretrain)')
ax1 = plt.subplot(1, 2, 1)
plot_image_with_graph(img1, kpts1, A1)
ax2 = plt.subplot(1, 2, 2)
plot_image_with_graph(img2, kpts2, A2)
for i in range(X.shape[0]):
    j = np.argmax(X[i]).item()
    con = ConnectionPatch(xyA=kpts1[:, i], xyB=kpts2[:, j], coordsA="data", coordsB="data",
                          axesA=ax1, axesB=ax2, color="red" if i != j else "green")
    plt.gca().add_artist(con)

In [ ]:
# calculate the matching accuracy
X_gt = np.diag(np.ones(kpts1.shape[1], dtype=np.float32))
(X * X_gt).sum() / X_gt.sum()

In [ ]:
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.title('Image 1')
plot_image_with_graph(img1, kpts1)
plt.subplot(1, 2, 2)
plt.title('Image 2')
plot_image_with_graph(img2, kpts2)

In [ ]:
gt_dict[(0,1)].todense()

In [ ]:
X

In [ ]:
import pygmtools as pygm
import numpy as np
import torch

# 1. 设置后端 (本示例使用 numpy)
pygm.set_backend('pytorch')
# setting random seed for reproducibility
np.random.seed(84)
torch.manual_seed(84)


import numpy as np
import time

def generate_kbqap_instance(n_nodes=128, delta_s=0.05, sigma_n=0.02):
    # 1. Create Ground Truth Points (sampled from 10K pool)
    # Sampling from U(0,1) x U(0,1)
    
    target_points = np.random.uniform(0, 1, (n_nodes, 2))
    
    # 2. Distort points
    # Random scaling s ~ U(1 - delta_s, 1 + delta_s)
    s = np.random.uniform(1 - delta_s, 1 + delta_s)
    # Gaussian noise epsilon ~ N(0, sigma_n^2)
    noise = np.random.normal(0, sigma_n, (n_nodes, 2))
    distorted_points = s * target_points + noise
    
    # 3. Create Reference Graph by randomly permuting node order
    permutation = np.random.permutation(n_nodes)
    reference_points = distorted_points[permutation]
    
    # 4. Define Similarity Function T_{i,j} = exp(-L2(c1, c2))
    def compute_similarity(pts_a, pts_b):
        # Calculate pairwise L2 distances
        diff = pts_a[:, np.newaxis, :] - pts_b[np.newaxis, :, :]
        dist_sq = np.sum(diff**2, axis=-1)
        return np.exp(-dist_sq)

    # Construct KBQAP Matrices
    # F1: Intra-similarity matrix for target graph
    F1 = compute_similarity(target_points, target_points)
    
    # F2: Intra-similarity matrix for reference graph
    F2 = compute_similarity(reference_points, reference_points)
    
    # Kp: Inter-similarity matrix between target and reference
    Kp = compute_similarity(target_points, reference_points)
    
    return F1, F2, Kp, permutation

# --- 开始测试 ---

# 1. 生成 GM-I 难度的单对数据
n = 128
F1_np, F2_np, Kp_np, gt_perm_np = generate_kbqap_instance(n_nodes=n, delta_s=0.05, sigma_n=0.02)
# gt_perm 转换为置换矩阵形式
gt_perm_matrix = np.zeros((n, n))
for i in range(n):
    gt_perm_matrix[i, gt_perm_np[i]] = 1
gt_perm = gt_perm_matrix

K = np.kron(F2_np, F1_np) + np.diag(Kp_np.T.flatten())

F1 = torch.tensor(F1_np, dtype=torch.float32)
F2 = torch.tensor(F2_np, dtype=torch.float32)
Kp = torch.tensor(Kp_np, dtype=torch.float32)
gt_perm = torch.tensor(gt_perm, dtype=torch.float32)
K = torch.tensor(K, dtype=torch.float32)

# 2. 构造亲和矩阵 K (Lawler Form)
# 许多传统的 pygmtools 求解器需要 K 矩阵。K = F2 ⊗ F1
# 注意：对于 128 节点的图，K 是 16384 x 16384，内存占用约 2GB (float64)

# 3. 定义要测试的算法
# 注意：IPFP 可以直接接受 KB 形式 (D, F)，效率更高
solvers = {
    'Spectral': lambda: pygm.sm(K, n, n),
    'RRWM': lambda: pygm.rrwm(K, n, n),
    'IPFP': lambda: pygm.ipfp(K, n, n) # 直接传入 F1, F2
}

print(f"{'Algorithm':<15} | {'Accuracy':<10} | {'Objective Score':<15}")
print("-" * 50)

for name, solver_func in solvers.items():
    try:
        # 运行求解器得到连续解 (Birkhoff Polytope)
        time0 = time.time()
        soft_X = solver_func()
        
        # 离散化投影到置换矩阵
        X_res = pygm.hungarian(soft_X)
        solve_time = time.time() - time0
        # 计算准确率
        # acc = pygm.utils.accuracy(X_res, gt_X)
        acc = (X_res * gt_perm).sum() / gt_perm.sum()
        
        # 计算目标函数值 tr(F2 * X * F1 * X.T)
        # score1 = np.trace(F1 @ X_res @ F2 @ X_res.T) + np.sum(Kp * X_res)
        # score1 = np.sum(np.multiply(X_res, np.matmul(F1, np.matmul(X_res, F2)))) + np.sum(np.multiply(X_res, Kp))
        score = pygm.utils.compute_affinity_score(X_res, K)
        
        print(f"{name:<15} | {acc:<10.4f} | {score:<15.4f} | {solve_time:<10.4f}")
        
    except Exception as e:
        print(f"{name:<15} | Error: {e}")

# 计算 Ground Truth 的目标分数值作为参考
# gt_score = np.trace(F2 @ gt_perm @ F1 @ gt_perm.T) + np.sum(Kp * gt_perm)
gt_score = pygm.utils.compute_affinity_score(gt_perm, K)
print("-" * 50)
print(f"{'Ground Truth':<15} | {1.0000:<10} | {gt_score:<15.4f}")

In [ ]:
F1.shape

In [ ]:
incumbent_X, incumbent_obj, total_time, incumbents, incum_time = run_optimization(F1_np, F2_np, Kp_np, 1, 20, num_steps=100)

In [ ]:
pygm.utils.compute_affinity_score(incumbent_X.to("cpu"), K)

In [1]:
import pygmtools as pygm
import numpy as np
import torch
import time

pygm.set_backend('pytorch')
np.random.seed(42)
torch.manual_seed(42)

from main import run_optimization

class GraphMatchingDatasetGenerator:
    def __init__(self, pool_size=10000):
        # 1. Create 10K ground truth points pool: U(0, 1) x U(0, 1)
        self.pool = np.random.uniform(0, 1, (pool_size, 2))
        
    def _compute_similarity(self, pts_a, pts_b):
        """Computes T_ij = exp(-L2(c1, c2))"""
        # Efficient pairwise L2 distance calculation
        diff = pts_a[:, np.newaxis, :] - pts_b[np.newaxis, :, :]
        dist_l2 = np.linalg.norm(diff, axis=-1)
        return np.exp(-dist_l2)

    def generate_instance(self, n_nodes=128, delta_s=0.05, sigma_n=0.02):
        # 2. Sample 128 nodes from the 10K pool
        indices = np.random.choice(len(self.pool), n_nodes, replace=False)
        target_pts = self.pool[indices]
        
        # 3. Perturb points to create reference set
        # Scaling: s ~ U(1 - delta_s, 1 + delta_s)
        s = np.random.uniform(1 - delta_s, 1 + delta_s)
        # Noise: epsilon ~ N(0, sigma_n^2)
        noise = np.random.normal(0, sigma_n, (n_nodes, 2))
        distorted_pts = s * target_pts + noise
        
        # 4. Randomly permute node order for reference graph
        perm_ground_truth = np.random.permutation(n_nodes)

        reference_pts = distorted_pts[perm_ground_truth]
        
        gt_perm_matrix = np.zeros((n_nodes, n_nodes))
        for i in range(n_nodes):
            gt_perm_matrix[i, perm_ground_truth[i]] = 1
        perm_ground_truth = gt_perm_matrix
        # 5. Construct KBQAP Matrices
        F1 = self._compute_similarity(target_pts, target_pts) # Intra-target
        F2 = self._compute_similarity(reference_pts, reference_pts) # Intra-reference
        Kp = self._compute_similarity(target_pts, reference_pts) # Inter-similarity
        
        return {
            "F1": F1, 
            "F2": F2, 
            "Kp": Kp, 
            "perm": perm_ground_truth,
            "target_pts": target_pts,
            "ref_pts": reference_pts
        }

    def generate_full_dataset(self, n_samples=200, n_nodes=128, config='GM-I'):
        # Parameter selection based on paper (GM-I vs GM-II)
        if config == 'GM-I':
            ds, sn = 0.05, 0.02
        else: # GM-II
            ds, sn = 0.3, 0.2
            
        print(f"Generating {config} dataset ({ds=}, {sn=})...")
        
        data = [self.generate_instance(n_nodes, ds, sn) for _ in range(n_samples)]
        
        return data

# --- Usage ---
generator = GraphMatchingDatasetGenerator()

# Generate GM-I (Easy) and GM-II (Hard) configurations
train_g1 = generator.generate_full_dataset(config='GM-I')
# train_g1 = generator.generate_full_dataset(config='GM-II')

for i, data in enumerate(train_g1):
    F1_np, F2_np, Kp_np, perm_np = data['F1'], data['F2'], data['Kp'], data['perm']
    K_np = np.kron(F2_np, F1_np) + np.diag(Kp_np.T.flatten())
    
    F1 = torch.tensor(F1_np, dtype=torch.float32)
    F2 = torch.tensor(F2_np, dtype=torch.float32)
    Kp = torch.tensor(Kp_np, dtype=torch.float32)
    gt_perm = torch.tensor(perm_np, dtype=torch.float32)
    K = torch.tensor(K_np, dtype=torch.float32)
    n = F1.shape[0]
    solvers = {
        'SM': lambda: pygm.sm(K, n, n),
        'RRWM': lambda: pygm.rrwm(K, n, n),
        'IPFP': lambda: pygm.ipfp(K, n, n),  # 直接传入 F1, F2
        'PDBO': lambda: run_optimization(F1_np, F2_np, Kp_np, 5, 50, num_steps=50)
    }
    print(i+1, "th instance results:")
    print(f"{'Algorithm':<15} | {'Accuracy':<10} | {'Objective Score':<15} | {'Time(s)':<10}")
    print("-" * 50)
    
    obj_results = {'SM': [], 'RRWM': [], 'IPFP': [], 'PDBO': [], 'GT': []}
    time_results = {'SM': [], 'RRWM': [], 'IPFP': [], 'PDBO': []}

    for name, solver_func in solvers.items():
        try:
            if name != 'PDBO':
                time0 = time.time()
                soft_X = solver_func()
                X_res = pygm.hungarian(soft_X)
                solve_time = time.time() - time0
            else:
                X_res, _, solve_time, _, _ = solver_func()
            acc = (X_res.to('cpu') * gt_perm).sum() / gt_perm.sum()
            score = pygm.utils.compute_affinity_score(X_res.to('cpu'), K)
            
            obj_results[name].append(score.item())
            time_results[name].append(solve_time)
            
            print(f"{name:<15} | {acc:<10.4f} | {score:<15.4f} | {solve_time:<10.4f}")
            
        except Exception as e:
            print(f"{name:<15} | Error: {e}")

    gt_score = pygm.utils.compute_affinity_score(gt_perm, K)
    print("-" * 50)
    print(f"{'Ground Truth':<15} | {1.0000:<10} | {gt_score:<15.4f}")
    
    obj_results['GT'].append(gt_score.item())
    print(i+1, "instances processed.")

# print the average results
print(f"{'Algorithm':<15} | {'Average Objective':<10} | {'Average Time(s)':<10}")
print("-" * 50)
for name in obj_results.keys():
    avg_obj = np.mean(obj_results[name])
    if name != 'GT':
        avg_time = np.mean(time_results[name])
        print(f"{name:<15} | {avg_obj:<10.4f} | {avg_time:<10.4f}")
    else:
        print(f"{name:<15} | {avg_obj:<10.4f}")
        

/home/xjx/miniconda3/envs/scopf/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Generating GM-I dataset (ds=0.05, sn=0.02)...
1 th instance results:
Algorithm       | Accuracy   | Objective Score | Time(s)   
--------------------------------------------------
SM              | 0.0156     | 6465.1187       | 0.1514    
RRWM            | 0.0234     | 6549.6963       | 1.3736    
IPFP            | 0.0312     | 6551.5654       | 0.3469    


/home/xjx/miniconda3/envs/scopf/lib/python3.10/site-packages/torch/_inductor/compile_fx.py:321: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


PDBO            | 0.0156     | 6562.4854       | 6.0111    
--------------------------------------------------
Ground Truth    | 1.0        | 6166.7988      
1 instances processed.
2 th instance results:
Algorithm       | Accuracy   | Objective Score | Time(s)   
--------------------------------------------------
SM              | 0.0078     | 6408.2266       | 0.1205    
RRWM            | 0.0234     | 6485.3667       | 1.3734    
IPFP            | 0.0078     | 6492.4873       | 0.3505    
PDBO            | 0.0156     | 6508.8184       | 0.4361    
--------------------------------------------------
Ground Truth    | 1.0        | 6113.4531      
2 instances processed.
3 th instance results:
Algorithm       | Accuracy   | Objective Score | Time(s)   
--------------------------------------------------
SM              | 0.0312     | 6563.8223       | 0.1171    
RRWM            | 0.0312     | 6622.6094       | 1.3942    
IPFP            | 0.0312     | 6645.2969       | 0.3514    
PDBO      

In [ ]:
incumbent_obj

In [ ]:
X_res = incumbent_X.detach().cpu().numpy()
np.sum(np.multiply(X_res, np.matmul(F2, np.matmul(X_res, F1)))) + np.sum(np.multiply(X_res, Kp))